In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_table= f"{catalog_name}.{bronze_schema}.constructors"
silver_table= f"{catalog_name}.{silver_schema}.constructors"

In [0]:
constructors_df = spark.table(bronze_table)

In [0]:
constructors_drop_df= constructors_df.drop("url")

In [0]:
constructors_rename_df= (constructors_drop_df
.withColumnRenamed("constructorId", "constructor_id")
.withColumnRenamed("name", "constructor_name")
.withColumn("batch_id", F.lit(v_batch_id))
)

In [0]:
display(constructors_rename_df)

In [0]:
constructors_dup_df = constructors_rename_df.dropDuplicates(["constructor_id"])

In [0]:
display(constructors_dup_df)

In [0]:
constructor_final_df=( constructors_dup_df.withColumn("nationality", F.initcap(F.col("nationality"))))

In [0]:
write_to_silver(
    input_df= constructor_final_df,
    target_table= silver_table,
    merge_condition= "t.constructor_id= s.constructor_id",
    columns_to_update=[
        "constructor_name",
        "nationality",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))